# MC Sim - GPU Build & Test

Build and test the molecular communication GPU simulator on Colab.

**Before running:** Go to Runtime > Change runtime type > T4 GPU

In [ ]:
# Verify GPU is available
!nvidia-smi
!nvcc --version

In [ ]:
# Clone the repo
!git clone https://github.com/alwaysEpic/molecular_modeling_gpu.git
%cd molecular_modeling_gpu

In [ ]:
# Build both targets
!mkdir -p build && cd build && cmake .. && make -j$(nproc)
!ls -la build/mc_sim build/mc_sim_cpu

## GPU Test - Quick Smoke Test

In [ ]:
# Quick GPU test - 1000 paths, first-hit, verbose
!cd build && ./mc_sim -i 1000 -f -v

## CPU vs GPU Comparison

In [ ]:
# CPU vs GPU comparison - 1000 paths
!cd build && ./mc_sim -i 1000 -c -f -v

In [ ]:
# CPU vs GPU comparison - 10000 paths
!cd build && ./mc_sim -i 10000 -c -f -v

## CPU-Only Benchmarks (for BENCHMARKS.md)

In [ ]:
# CPU-only timing - matches thesis benchmark parameters
!cd build && ./mc_sim_cpu -i 1000 -f -v
print("---")
!cd build && ./mc_sim_cpu -i 1000 -f -v -n

## Validation - 1D First-Hit with Drift

Compares simulation output against analytical inverse Gaussian (thesis eq 4.3).

In [ ]:
# Install validation deps
!pip install -q numpy matplotlib

In [ ]:
# Run GPU simulation for validation (10k paths, 1D limit, drift)
!cd build && ./mc_sim -i 10000 -f -l 3E-7 -t 1E-2 -v

In [ ]:
# Validate GPU output against analytical solution
!python scripts/validate_1d_firsthit.py build/output_d_wide.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2

In [ ]:
# Run CPU simulation for validation comparison
!cd build && ./mc_sim_cpu -i 10000 -f -l 3E-7 -t 1E-2 -v

In [ ]:
# Validate CPU output
!python scripts/validate_1d_firsthit.py build/output_h.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2